In [3]:
import os
import csv
import shutil

def copy_folder(source_path, folder_path):
    try:
        # フォルダがすでに存在する場合、削除してからコピーを行う
        if os.path.exists(folder_path):
            shutil.rmtree(folder_path)
        shutil.copytree(source_path, folder_path)
        print(f"フォルダをコピーしました。")
    except shutil.Error as e:
        print(f"エラー: {e}")
    except OSError as e:
        print(f"エラー: {e}")

def rename_files(csv_file_path, folder_path):
    # CSVファイルを読み込む
    with open(csv_file_path, 'r', encoding='shift_jis') as csv_file:  # 必要に応じてエンコーディングを変更
        csv_reader = csv.reader(csv_file)
        next(csv_reader)  # ヘッダー行をスキップ

        # CSVにあるファイル名のリストを作成
        valid_filenames = set()

        # フォルダ内の.wavファイルを変換
        for index, row in enumerate(csv_reader):
            old_filename = row[0]
            new_filename = row[2]
            valid_filenames.add(old_filename)

            # 拡張子を付けてファイルパスを作成
            old_filepath = os.path.join(folder_path, old_filename + '.wav')
            new_filepath = os.path.join(folder_path, 'index=' + str(index+1) + '.' + new_filename + '.wav')

            # ファイルのリネーム
            try:
                os.rename(old_filepath, new_filepath)
                print(f'Renamed: {old_filename}.wav to {new_filename}.wav')
            except FileNotFoundError:
                print(f'File not found: {old_filename}.wav')
            except Exception as e:
                print(f'Error renaming file {old_filename}.wav: {e}')

        # CSVに含まれていないファイルを削除
        for filename in os.listdir(folder_path):
            if filename.endswith('.wav'):
                base_filename = os.path.splitext(filename)[0]  # 拡張子を除いたファイル名
                if base_filename not in valid_filenames and not base_filename.startswith('index='):
                    file_to_remove = os.path.join(folder_path, filename)
                    os.remove(file_to_remove)
                    print(f'Removed file: {filename}')

# CSVファイルとフォルダのパスを指定
csv_file_path = r"C:\Users\Casper4\Python\ueki\shibasaki\研究\Pool_boiling\Subcooling_20_degrees\0.3\2025.07.09_0.3_1\実験結果2025.07.09_0.3_1\heat_flux_2025.07.09_0.3_1.csv"

# 変換用にコピーフォルダを作成
source_path = os.path.join(os.path.dirname(os.path.dirname(csv_file_path)), '録音データ')  # 1つ上の階層
folder_path = os.path.join(os.path.dirname(source_path), '録音データ_熱流束')
copy_folder(source_path, folder_path)

# ファイルをリネーム
rename_files(csv_file_path, folder_path)


フォルダをコピーしました。
Renamed: 0.0V.wav to 6.98E-05.wav
Renamed: 0.1V.wav to 2.42E+03.wav
Renamed: 0.3V.wav to 2.16E+04.wav
Renamed: 0.5V.wav to 5.94E+04.wav
Renamed: 0.7V.wav to 1.15E+05.wav
Renamed: 0.9V.wav to 1.87E+05.wav
Renamed: 1.0V.wav to 2.29E+05.wav
Renamed: 1.1V.wav to 2.75E+05.wav
Renamed: 1.2V.wav to 3.27E+05.wav
Renamed: 1.3V.wav to 3.83E+05.wav
Renamed: 1.4V.wav to 4.42E+05.wav
Renamed: 1.5V.wav to 5.05E+05.wav
Renamed: 1.6V.wav to 5.72E+05.wav
Renamed: 1.7V.wav to 6.44E+05.wav
Renamed: 1.8V.wav to 7.21E+05.wav
Renamed: 1.9V.wav to 8.02E+05.wav
Renamed: 2.0V.wav to 8.88E+05.wav


# 熱流束を丸め込まないバージョン

In [4]:
import os
import csv
import shutil

def copy_folder(source_path, folder_path):
    try:
        # フォルダがすでに存在する場合、削除してからコピーを行う
        if os.path.exists(folder_path):
            shutil.rmtree(folder_path)
        shutil.copytree(source_path, folder_path)
        print(f"フォルダをコピーしました。")
    except shutil.Error as e:
        print(f"エラー: {e}")
    except OSError as e:
        print(f"エラー: {e}")

def rename_files(csv_file_path, folder_path):
    # CSVファイルを読み込む
    with open(csv_file_path, 'r', encoding='shift_jis') as csv_file:  # 必要に応じてエンコーディングを変更
        csv_reader = csv.reader(csv_file)
        next(csv_reader)  # ヘッダー行をスキップ

        # CSVにあるファイル名のリストを作成
        valid_filenames = set()

        # フォルダ内の.wavファイルを変換
        for index, row in enumerate(csv_reader):
            old_filename = row[0] # 1列目 (volt) を取得
            
            # --- 変更箇所: ここから ---
            # 元のコード: new_filename = row[2]
            # 修正後の論理: 2列目(q)を取得し、数値変換 -> 整数化(切り捨て)を行う
            try:
                q_value_str = row[1]          # 2列目の値を取得 (例: "2424.755")
                q_value_float = float(q_value_str) # 小数に変換 (2424.755)
                q_value_int = int(q_value_float)   # 整数に変換して切り捨て (2424)
                new_filename = str(q_value_int)    # 文字列に戻す ("2424")
            except ValueError:
                print(f"警告: 行 {index+1} の値 '{row[1]}' は数値に変換できませんでした。スキップします。")
                continue
            # --- 変更箇所: ここまで ---

            valid_filenames.add(old_filename)

            # 拡張子を付けてファイルパスを作成
            old_filepath = os.path.join(folder_path, old_filename + '.wav')
            new_filepath = os.path.join(folder_path, 'index=' + str(index+1) + '.' + new_filename + '.wav')

            # ファイルのリネーム
            try:
                os.rename(old_filepath, new_filepath)
                print(f'Renamed: {old_filename}.wav to {new_filename}.wav (Value: {row[1]} -> {new_filename})')
            except FileNotFoundError:
                print(f'File not found: {old_filename}.wav')
            except Exception as e:
                print(f'Error renaming file {old_filename}.wav: {e}')

        # CSVに含まれていないファイルを削除
        for filename in os.listdir(folder_path):
            if filename.endswith('.wav'):
                # 拡張子を除いたファイル名
                base_filename = os.path.splitext(filename)[0] 
                
                # index=で始まるファイル（変換済み）は削除対象外とする
                if base_filename.startswith('index='):
                    continue
                
                # 元のファイル名リストになければ削除
                if base_filename not in valid_filenames:
                    file_to_remove = os.path.join(folder_path, filename)
                    try:
                        os.remove(file_to_remove)
                        print(f'Removed file: {filename}')
                    except Exception as e:
                        print(f'Error removing file {filename}: {e}')

# CSVファイルとフォルダのパスを指定
csv_file_path = r"C:\Users\Casper4\Python\ueki\shibasaki\研究\Pool_boiling\Subcooling_20_degrees\0.3\2025.07.09_0.3_1\実験結果2025.07.09_0.3_1\heat_flux_2025.07.09_0.3_1.csv"

# 変換用にコピーフォルダを作成
source_path = os.path.join(os.path.dirname(os.path.dirname(csv_file_path)), '録音データ')  # 1つ上の階層
folder_path = os.path.join(os.path.dirname(source_path), '録音データ_熱流束')
copy_folder(source_path, folder_path)

# ファイルをリネーム
rename_files(csv_file_path, folder_path)

フォルダをコピーしました。
Renamed: 0.0V.wav to 0.wav (Value: 6.98E-05 -> 0)
Renamed: 0.1V.wav to 2424.wav (Value: 2424.754654 -> 2424)
Renamed: 0.3V.wav to 21613.wav (Value: 21613.23193 -> 21613)
Renamed: 0.5V.wav to 59375.wav (Value: 59375.42091 -> 59375)
Renamed: 0.7V.wav to 115043.wav (Value: 115043.3316 -> 115043)
Renamed: 0.9V.wav to 187171.wav (Value: 187171.717 -> 187171)
Renamed: 1.0V.wav to 229386.wav (Value: 229386.873 -> 229386)
Renamed: 1.1V.wav to 275174.wav (Value: 275174.6641 -> 275174)
Renamed: 1.2V.wav to 327124.wav (Value: 327124.6689 -> 327124)
Renamed: 1.3V.wav to 383287.wav (Value: 383287.3546 -> 383287)
Renamed: 1.4V.wav to 442169.wav (Value: 442169.2924 -> 442169)
Renamed: 1.5V.wav to 505101.wav (Value: 505101.2689 -> 505101)
Renamed: 1.6V.wav to 571694.wav (Value: 571694.2525 -> 571694)
Renamed: 1.7V.wav to 643516.wav (Value: 643516.9403 -> 643516)
Renamed: 1.8V.wav to 720690.wav (Value: 720690.9057 -> 720690)
Renamed: 1.9V.wav to 802409.wav (Value: 802409.6877 -> 802409)
R

In [5]:
#深層学習で使う場合のラベルの作成方法
text = os.path.splitext(os.path.basename(os.path.join((folder_path), 'index=27.1.02e+06.wav')))[0]

dot_index = text.find(".")  #最初に出現するドットのインデックスを取得
if dot_index != -1:  #ドットの存在で判定
    extracted_label = text[dot_index+1:]  #1つ目のドット後の文字列を抽出
    print(f"Original textname: {text}, Extracted label: {extracted_label}")

Original textname: index=27.1.02e+06, Extracted label: 1.02e+06
